In [28]:
import pandas as pd

df = pd.read_csv("/home/pratyush_device/Documents/college/AIML lab/Project/customer.csv")

df.head()

,customer_id,acquisition_date,acquisition_cost_usd,market_segment,supplier_id,order_id,order_date,order_value_usd,payment_date,satisfaction_score,support_tickets,lead_time_days
0,CUST-001,2024-01-05,850,North America,SUP-A,ORD-1001,2024-01-15,12500,2024-02-18,4,1,14
1,CUST-002,2024-01-06,920,Europe,SUP-B,ORD-1002,2024-01-18,8500,2024-02-25,5,0,16
2,CUST-003,2024-01-07,880,Asia-Pacific,SUP-C,ORD-1003,2024-01-20,21000,2024-03-01,4,2,15
3,CUST-004,2024-01-08,790,North America,SUP-A,ORD-1004,2024-01-22,7500,2024-02-28,3,1,13
4,CUST-005,2024-01-09,950,Europe,SUP-D,ORD-1005,2024-01-25,15000,2024-03-05,5,0,12


In [29]:
df.isnull().sum()
df = df.drop("customer_id" , axis = 1)

In [30]:
df

,acquisition_date,acquisition_cost_usd,market_segment,supplier_id,order_id,order_date,order_value_usd,payment_date,satisfaction_score,support_tickets,lead_time_days
0,2024-01-05,850,North America,SUP-A,ORD-1001,2024-01-15,12500,2024-02-18,4,1,14
1,2024-01-06,920,Europe,SUP-B,ORD-1002,2024-01-18,8500,2024-02-25,5,0,16
2,2024-01-07,880,Asia-Pacific,SUP-C,ORD-1003,2024-01-20,21000,2024-03-01,4,2,15
3,2024-01-08,790,North America,SUP-A,ORD-1004,2024-01-22,7500,2024-02-28,3,1,13
4,2024-01-09,950,Europe,SUP-D,ORD-1005,2024-01-25,15000,2024-03-05,5,0,12
...,...,...,...,...,...,...,...,...,...,...,...
745,2026-01-20,880,Asia-Pacific,SUP-E,ORD-1746,2029-04-02,20500,2029-05-12,3,1,16
746,2026-01-21,860,Middle East,SUP-B,ORD-1747,2029-04-05,9000,2029-05-15,4,1,15
747,2026-01-22,900,North America,SUP-D,ORD-1748,2029-04-08,20000,2029-05-18,5,0,12
748,2026-01-23,850,Europe,SUP-F,ORD-1749,2029-04-10,15000,2029-05-20,4,2,14


In [31]:

df = df.drop("order_id" , axis = 1)
df = df.drop("acquisition_date" , axis = 1)
df = df.drop("order_date" , axis = 1)
df = df.drop("payment_date" , axis = 1)


In [32]:
df.shape

(750, 7)

In [33]:
df


,acquisition_cost_usd,market_segment,supplier_id,order_value_usd,satisfaction_score,support_tickets,lead_time_days
0,850,North America,SUP-A,12500,4,1,14
1,920,Europe,SUP-B,8500,5,0,16
2,880,Asia-Pacific,SUP-C,21000,4,2,15
3,790,North America,SUP-A,7500,3,1,13
4,950,Europe,SUP-D,15000,5,0,12
...,...,...,...,...,...,...,...
745,880,Asia-Pacific,SUP-E,20500,3,1,16
746,860,Middle East,SUP-B,9000,4,1,15
747,900,North America,SUP-D,20000,5,0,12
748,850,Europe,SUP-F,15000,4,2,14


In [34]:
from sklearn.preprocessing import OneHotEncoder

enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded = enc.fit_transform(df[['market_segment']])

encoded_df = pd.DataFrame(
    encoded,
    columns=enc.get_feature_names_out(['market_segment'])
)

df = pd.concat([df.drop('market_segment', axis=1), encoded_df], axis=1)

In [35]:

mean_map = df.groupby('supplier_id')['order_value_usd'].mean()


df['supplier_id'] = df['supplier_id'].map(mean_map)

In [36]:
df

,acquisition_cost_usd,supplier_id,order_value_usd,satisfaction_score,support_tickets,lead_time_days,market_segment_Africa,market_segment_Asia-Pacific,market_segment_Europe,market_segment_Middle East,market_segment_North America
0,850,14944.444444,12500,4,1,14,0.0,0.0,0.0,0.0,1.0
1,920,15074.074074,8500,5,0,16,0.0,0.0,1.0,0.0,0.0
2,880,15303.738318,21000,4,2,15,0.0,1.0,0.0,0.0,0.0
3,790,14944.444444,7500,3,1,13,0.0,0.0,0.0,0.0,1.0
4,950,15411.214953,15000,5,0,12,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
745,880,15056.074766,20500,3,1,16,0.0,1.0,0.0,0.0,0.0
746,860,15074.074074,9000,4,1,15,0.0,0.0,0.0,1.0,0.0
747,900,15411.214953,20000,5,0,12,0.0,0.0,0.0,0.0,1.0
748,850,15074.766355,15000,4,2,14,0.0,0.0,1.0,0.0,0.0


In [37]:
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split

x = df.drop("lead_time_days" , axis = 1)
y = df["lead_time_days"]


In [38]:
x_train , x_test ,y_train , y_test = train_test_split(x, y , test_size = 0.3 , random_state = 40)

from sklearn.preprocessing import StandardScaler

scalar = StandardScaler()

x_train_scaled = scalar.fit_transform(x_train)
x_test_scaled = scalar.transform(x_test)

In [39]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

model1 = LinearRegression()
model2 = RandomForestRegressor()
model3 = SVR()


model1.fit(x_train_scaled, y_train)


LinearRegression()

In [40]:
model2.fit(x_train_scaled , y_train)


RandomForestRegressor()

In [41]:
model3.fit(x_train_scaled , y_train)

SVR()

In [42]:
y_pred1 = model1.predict(x_test_scaled)

In [43]:
y_pred2 = model2.predict(x_test_scaled)

In [44]:
y_pred3 = model3.predict(x_test_scaled)

In [45]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

MAE = mean_absolute_error(y_test , y_pred1)
MSE = mean_squared_error(y_test , y_pred1)
R2_score = r2_score(y_test , y_pred1)

print(MAE)
print(R2_score)

0.43216093657718274
0.8985268216639708


In [46]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

MAE = mean_absolute_error(y_test , y_pred2)
MSE = mean_squared_error(y_test , y_pred2)
R2_score = r2_score(y_test , y_pred2)

print(MAE)
print(R2_score)

0.21203777777777766
0.932301096145463


In [47]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

MAE = mean_absolute_error(y_test , y_pred3)
MSE = mean_squared_error(y_test , y_pred3)
R2_score = r2_score(y_test , y_pred3)

print(MAE)
print(R2_score)

0.27423038833802216
0.9224068116970618
